# Day 2
## Task 1 Preprocessing Plan & Implementation
The goal of this task is to build a reproducible preprocessing pipeline for the Adult Income dataset.
The dataset contains both numerical and categorical features. A ColumnTransformer` will be used to apply appropriate preprocessing to each feature type while preventing data leakage by fitting preprocessing steps only on the training data.

In [7]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
adult = fetch_openml(
    "adult",
    version=2,
    as_frame=True
)

df = adult.frame.copy()

print("Dataset shape:", df.shape)

df = df.replace("?", np.nan)
print("Missing values converted to NaN.")

df["target"] = df["class"].map({
    "<=50K": 0,
    ">50K": 1
})

print(df["target"].value_counts())

X = df.drop(columns=["class", "target"])
y = df["target"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Dataset shape: (48842, 15)
Missing values converted to NaN.
target
0    37155
1    11687
Name: count, dtype: int64
Feature shape: (48842, 14)
Target shape: (48842,)


In [2]:
numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

print("Numeric features:")
for feature in numeric_features:
    print("-", feature)

Numeric features:
- age
- fnlwgt
- education-num
- capital-gain
- capital-loss
- hours-per-week


In [5]:
categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

print("Categorical features:")
for feature in categorical_features:
    print("-", feature)

Categorical features:
- workclass
- education
- marital-status
- occupation
- relationship
- race
- sex
- native-country


In [8]:
# Verify that every feature has been assigned to exactly one group

all_features = numeric_features + categorical_features

print("Total features:", len(all_features))
print("Features in X:", len(X.columns))

print(
    "All features covered:",
    set(all_features) == set(X.columns)
)

Total features: 14
Features in X: 14
All features covered: True


In [9]:
# Numeric preprocessing:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print("Numeric pipeline created successfully!")

Numeric pipeline created successfully!


In [10]:
# Categorical preprocessing:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

print("Categorical pipeline created successfully!")

Categorical pipeline created successfully!


In [11]:
# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

print("ColumnTransformer created successfully!")

ColumnTransformer created successfully!


In [12]:
# Recreate the same 20% stratified hold-out test split used on Day 1
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Development split from the remaining data

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev,
    y_train_dev,
    test_size=0.125,
    random_state=42,
    stratify=y_train_dev
)

print("Training:", X_train.shape)
print("Development:", X_dev.shape)
print("Hold-out Test:", X_test.shape)

Training: (34188, 14)
Development: (4885, 14)
Hold-out Test: (9769, 14)


In [13]:
# Fit the preprocessing pipeline ONLY on the training data
X_train_processed = preprocessor.fit_transform(X_train)
# Transform development and test data using the already-fitted preprocessor
X_dev_processed = preprocessor.transform(X_dev)
X_test_processed = preprocessor.transform(X_test)
print("Training processed shape:", X_train_processed.shape)
print("Development processed shape:", X_dev_processed.shape)
print("Test processed shape:", X_test_processed.shape)

Training processed shape: (34188, 105)
Development processed shape: (4885, 105)
Test processed shape: (9769, 105)


In [18]:
# Get the names of all transformed features
feature_names = preprocessor.get_feature_names_out()
print("Number of processed features:", len(feature_names))
print("\nFirst 20 processed features:")
print(feature_names[:20])

# Verify that preprocessing removed missing values
print(
    "Missing values after preprocessing:",
    np.isnan(X_train_processed.toarray()).sum()
    if hasattr(X_train_processed, "toarray")
    else np.isnan(X_train_processed).sum()
)
# Save transformed feature names for later interpretability analysis
from pathlib import Path

feature_names_df = pd.DataFrame({
    "feature_name": feature_names
})

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

feature_names_df.to_csv(
    output_dir / "processed_feature_names.csv",
    index=False
)

print("Processed feature names saved successfully!")

Number of processed features: 105

First 20 processed features:
['numeric__age' 'numeric__fnlwgt' 'numeric__education-num'
 'numeric__capital-gain' 'numeric__capital-loss' 'numeric__hours-per-week'
 'categorical__workclass_Federal-gov' 'categorical__workclass_Local-gov'
 'categorical__workclass_Never-worked' 'categorical__workclass_Private'
 'categorical__workclass_Self-emp-inc'
 'categorical__workclass_Self-emp-not-inc'
 'categorical__workclass_State-gov' 'categorical__workclass_Without-pay'
 'categorical__education_10th' 'categorical__education_11th'
 'categorical__education_12th' 'categorical__education_1st-4th'
 'categorical__education_5th-6th' 'categorical__education_7th-8th']
Missing values after preprocessing: 0
Processed feature names saved successfully!


In [20]:
numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

#  Numerical preprocessing pipeline
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

#  Categorical preprocessing pipeline
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)
# Combine both pipelines using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)
print("\nColumnTransformer created successfully!")

#  Recreate the SAME split 
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
# Split the remaining 80% into approximately:
# 70% training and 10% development.
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev,
    y_train_dev,
    test_size=0.125,
    random_state=42,
    stratify=y_train_dev
)
#  Display split sizes
print("\nData Split:")
print("Training:", X_train.shape)
print("Development:", X_dev.shape)
print("Hold-out Test:", X_test.shape)

# Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_dev_processed = preprocessor.transform(X_dev)
X_test_processed = preprocessor.transform(X_test)


#  Display processed data shapes
print("\nProcessed Data:")
print("Training processed shape:", X_train_processed.shape)
print("Development processed shape:", X_dev_processed.shape)
print("Test processed shape:", X_test_processed.shape)

# Get names of transformed features
feature_names = preprocessor.get_feature_names_out()
print("\nNumber of processed features:", len(feature_names))
print("\nFirst 20 processed feature names:")
print(feature_names[:20])

#  Verify that preprocessing removed missing values
if hasattr(X_train_processed, "toarray"):
    missing_after_preprocessing = np.isnan(
        X_train_processed.toarray()
    ).sum()
else:
    missing_after_preprocessing = np.isnan(
        X_train_processed
    ).sum()

print(
    "\nMissing values after preprocessing:",
    missing_after_preprocessing
)

# Save processed feature names for later tasks

import os
os.makedirs("../outputs", exist_ok=True)
feature_names_df = pd.DataFrame({
    "feature_name": feature_names
})
feature_names_df.to_csv(
    "../outputs/processed_feature_names.csv",
    index=False
)
print("\nProcessed feature names saved successfully!")

Numerical features:
['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Categorical features:
['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

ColumnTransformer created successfully!

Data Split:
Training: (34188, 14)
Development: (4885, 14)
Hold-out Test: (9769, 14)

Processed Data:
Training processed shape: (34188, 105)
Development processed shape: (4885, 105)
Test processed shape: (9769, 105)

Number of processed features: 105

First 20 processed feature names:
['numeric__age' 'numeric__fnlwgt' 'numeric__education-num'
 'numeric__capital-gain' 'numeric__capital-loss' 'numeric__hours-per-week'
 'categorical__workclass_Federal-gov' 'categorical__workclass_Local-gov'
 'categorical__workclass_Never-worked' 'categorical__workclass_Private'
 'categorical__workclass_Self-emp-inc'
 'categorical__workclass_Self-emp-not-inc'
 'categorical__workclass_State-gov' 'categorical__workclass_Without-pay'
 'c

## Task 1 — Conclusion
The preprocessing pipeline for the Adult Income dataset was successfully implemented using a ColumnTransformer The numerical features were processed using median imputation followed by StandardScaler, while categorical features were processed using most-frequent imputation followed by OneHotEncoder(handle_unknown="ignore").

Median imputation was selected because it is more robust to outliers and skewed numerical distributions than mean imputation. One-hot encoding was selected because the categorical features do not have a natural numerical ordering. Other approaches such as mean imputation, KNN imputation, iterative imputation, ordinal encoding, and target encoding were considered but skipped because they were either less suitable, more complex, or could introduce additional leakage risks at this stage.

The preprocessing was fitted only on the training data and then applied to the development and hold-out test sets. This ensures that information from the evaluation data does not influence the preprocessing process and helps prevent data leakage. Therefore, the preprocessing pipeline is suitable for use with the supervised learning models in the next tasks.
